# Modélisation du risque de défaut TPE/PME


**Sommaire**
1. Chargement et fusion des sources
2. Construction de la variable Y de défaut
3. Suppression des data leak
4. Traitement des dates
5. Normalisation des unités
6. Blocs de valeurs manquantes (associé / financier)
7. Ratios financiers RA1–RA20
8. Encodage des variables catégorielles
9. Protocole d'évaluation
10. Comparaison des quatre modèles
11. Recherche d'hyperparamètres
12. Modèle final et importance des variables

In [115]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "1"

In [116]:

import pandas as pd
import numpy as np
import warnings
import itertools
import torch
import torch.nn as nn
warnings.filterwarnings('ignore')

from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,precision_score, recall_score, f1_score, roc_curve,confusion_matrix)
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from xgboost import XGBClassifier

torch.manual_seed(42)
np.random.seed(42)


## 1. Chargement et fusion des données



In [117]:
prod   = pd.read_table("Prod dev.txt",           encoding='cp1252', low_memory=False)
impaye = pd.read_table("Impayé dev.txt",        low_memory=False)
mej    = pd.read_table("MEJ dev.txt",            low_memory=False)
ind    = pd.read_table("Ind financiers dev.txt", encoding='cp1252', low_memory=False)

print("prod  :", prod.shape)
print("impayé:", impaye.shape, ":", impaye['IDENTIFIANT_CREDIT'].nunique(), "crédits distincts")
print("mej   :", mej.shape,    ":", mej['IDENTIFIANT_CREDIT'].nunique(), "crédits distincts")
print("ind   :", ind.shape,    ":", ind['CODE_INDICATEUR'].nunique(), "indicateurs")

prod  : (846, 22)
impayé: (267, 4) : 138 crédits distincts
mej   : (30, 9) : 27 crédits distincts
ind   : (229989, 8) : 54 indicateurs


In [118]:
imp_agg = impaye[["IDENTIFIANT_CREDIT","NBRE_JOUR_IMPAYE"]].groupby("IDENTIFIANT_CREDIT").sum().reset_index()
mej_u   = mej[["IDENTIFIANT_CREDIT","MONTANT_DEMANDE_MEJ"]].drop_duplicates(subset=["IDENTIFIANT_CREDIT"])
defaut  = pd.merge(imp_agg, mej_u, on="IDENTIFIANT_CREDIT", how="outer")

df = pd.merge(prod, defaut, on="IDENTIFIANT_CREDIT", how="left")
print(df.shape)

(846, 24)


### indicateurs financiers

Deux exercices sont disponibles pour la majorité des tiers (312 sur 467 en ont deux, 153 un
seul). Les moyenner écraserait toute dynamique. On retient donc le **dernier exercice** comme
niveau, complété par la **variation relative** entre premier et dernier exercice.

In [119]:
ind["DATEEXER"] = pd.to_datetime(ind["DATEEXER"], dayfirst=True, errors="coerce")
ind["VALEUR_FINALE"] = pd.to_numeric(ind["VALEUR_FINALE"].astype(str).str.replace(",", "."), errors="coerce")
ind = ind[ind["DATEEXER"].dt.year <= 2019]          # point d'observation : 31/12/2019
ind["IDENTIFIANT_TIERS"] = ind["IDENTIFIANT_TIERS"].astype(str).str.strip()
ind["AN"] = ind["DATEEXER"].dt.year

piv = ind.pivot_table(index=["IDENTIFIANT_TIERS","AN"], columns="CODE_INDICATEUR",
                      values="VALEUR_FINALE", aggfunc="mean")

ordre = piv.reset_index().sort_values("AN")
last  = ordre.groupby("IDENTIFIANT_TIERS").tail(1).set_index("IDENTIFIANT_TIERS").drop(columns="AN")
prev  = ordre.groupby("IDENTIFIANT_TIERS").head(1).set_index("IDENTIFIANT_TIERS").drop(columns="AN")

common = [c for c in last.columns if c in prev.columns]
var = ((last[common] - prev[common]) / prev[common].abs().replace(0, np.nan)) * 100
var.columns = ["VAR_" + c for c in var.columns]
var = var.clip(-300, 300)                          # clip des variations aberrantes

nb_ex = ordre.groupby("IDENTIFIANT_TIERS")["AN"].nunique().rename("NB_EXERCICES")
ind_wide = last.join(var).join(nb_ex)

df["IDENTIFIANT_TIERS"] = df["IDENTIFIANT_TIERS"].astype(str).str.strip()
df = pd.merge(df, ind_wide, on="IDENTIFIANT_TIERS", how="left")
print(df.shape, ":", len(var.columns), "variables de variation")

(846, 133) : 54 variables de variation


## 2. Variable cible

$$Y_i = \mathbf{1}\{\tau_i \le t+h\},\qquad Y_i = 1 \iff (\text{arriéré}) \lor (\text{mise en jeu de garantie})$$

In [120]:
df["DEFAUT"] = (df["NBRE_JOUR_IMPAYE"].notna() | df["MONTANT_DEMANDE_MEJ"].notna()).astype(int)
print(df["DEFAUT"].value_counts(), "\ntaux de défaut :", round(df["DEFAUT"].mean(), 4))

DEFAUT
0    690
1    156
Name: count, dtype: int64 
taux de défaut : 0.1844


## 3. Suppression des variables de fuite

| Variable | Manquants | Raison |
|---|---|---|
| `NBRE_JOUR_IMPAYE` | 84 % | sert à construire la cible |
| `MONTANT_DEMANDE_MEJ` | 97 % | idem |
| `CLASSE_RISQUE` | — | notation du modèle interne |
| `FINALITE_CREDIT` | — | une seule modalité, variance nulle |
| identifiants | — | aucun pouvoir prédictif |

In [121]:
drop = ["NBRE_JOUR_IMPAYE","MONTANT_DEMANDE_MEJ","CLASSE_RISQUE","FINALITE_CREDIT",
        "IDENTIFIANT_TIERS","IDENTIFIANT_CREDIT"]
df = df.drop(columns=[c for c in drop if c in df.columns])
print(df.shape)

(846, 128)


## 4. Ingénierie des dates

Une date brute n'a pas de sens on utilise des durées à la place

la date de déblocage implique que le crédit a été accordé et donc est une fuite de donnée



In [122]:
for c in ["DATE_CREATION_ENTREPRISE","DATE_ACCORD","DATE_PREMIER_DEBLOCAGE"]:
    df[c] = pd.to_datetime(df[c], format="%d/%m/%Y", errors="coerce")
    print(f"{c} manquants : {df[c].isna().mean():.1%}")

df["AGE_ENTREPRISE_ANS"] = (df["DATE_ACCORD"] - df["DATE_CREATION_ENTREPRISE"]).dt.days / 365.25
df["MOIS_ACCORD"]        = df["DATE_ACCORD"].dt.month
df["ANNEE_ACCORD"]       = df["DATE_ACCORD"].dt.year
df = df.drop(columns=["DATE_CREATION_ENTREPRISE","DATE_ACCORD","DATE_PREMIER_DEBLOCAGE"])

DATE_CREATION_ENTREPRISE manquants : 3.1%
DATE_ACCORD manquants : 0.0%
DATE_PREMIER_DEBLOCAGE manquants : 45.2%


## 5. Normalisation des unités de durées

`UNITE_DUREE_CONCOURS` prend deux modalités en mois ou en année
La périodicité reçoit un encodage **ordinal** qu'il faut mixer avec la durée du concours pour normaliser la variable.

Les 8 dossiers sans taux ou sans périodicité sont **supprimés** : ce sont des termes
contractuels, les imputer reviendrait à inventer les conditions du prêt.

In [123]:
df["DUREE_CONCOURS_MOIS"] = df["DUREE_CONCOURS"] * df["UNITE_DUREE_CONCOURS"].map({"mois":1,"an(s)":12})
df["PERIODICITE_MOIS"]    = df["PERIODICITE"].map({"Mensuelle":1,"Trimestrielle":3,"Quadrimestrielle":4,"Semestrielle":6,"Annuelle":12})

df["FLAG_PERIODICITE_NA"] = df["PERIODICITE"].isna().astype(int)
df["NBRE_ECHEANCES"]      = df["DUREE_CONCOURS_MOIS"] / df["PERIODICITE_MOIS"]

df = df.drop(columns=["DUREE_CONCOURS","UNITE_DUREE_CONCOURS","PERIODICITE"])

n0 = len(df)
df = df.dropna(subset=["TAUX_INTERET","PERIODICITE_MOIS"]).reset_index(drop=True)
print(f" {len(df)} lignes ({n0-len(df)} supprimées), taux de défaut : {df['DEFAUT'].mean():.4f}")

 838 lignes (8 supprimées), taux de défaut : 0.1862


## 6. Les deux blocs de valeurs manquantes

Deux groupes de colonnes manquent **par blocs**, pas au hasard.
Une information manquante dans le bloc dirigeant n'est jamais seule tout le bloc manque, idem pour les données de bilans financiers.

**Traitements différents :**

| Bloc | Nature | Imputation | Pourquoi |
|---|---|---|---|
| Associé (28 var.) | scores bornés | **0** | 0 = « non applicable » ; une médiane inventerait un profil de dirigeant |
| Financier (26 var.) | postes de bilan | **KNN (k=5)** | un actif à 0 signifierait une entreprise sans activité |

In [ ]:
bloc_associe = ['AGE1','AGE2','AGE3','ANTP1','ANTP2','ANTP3','ENGA1','ENGA2','ENGA3','EXP1','EXP2',
                'EXP3','LFAS1','LFAS2','LFAS3','LOCALE','NIVED1','NIVED2','NIVED3','NOTE11','NOTE12',
                'NOTE13','NOTE14','PATR1','PATR2','PATR3','QALDE','RGI']

bloc_fin = ['CU','CV','CW','DO','DU','ET','FF','FI','GI2','GI3','HA1','HA3','HA4','HA5','HDG','HP',
            'HZ','JCX','KW','LP','LV','MC','MD','MP','MT','MU']

bloc_associe = [c for c in bloc_associe if c in df.columns]
bloc_fin     = [c for c in bloc_fin     if c in df.columns]

Colonnes du bloc associé manquant sur les mêmes lignes : True
Lignes concernées : 0
Taux de défaut  bloc absent : nan | bloc présent : 0.186


In [ ]:
df["FLAG_PAS_ASSOCIE"]   = df[bloc_associe[0]].isna().astype(int)
df["FLAG_PAS_FINANCIER"] = df[bloc_fin[0]].isna().astype(int)
df[bloc_associe] = df[bloc_associe].fillna(0)
df["NB_EXERCICES"] = df["NB_EXERCICES"].fillna(0)

# KNN sur données normalisées 
sc_knn = StandardScaler()
Z = pd.DataFrame(sc_knn.fit_transform(df[bloc_fin]), columns=bloc_fin, index=df.index)
Z = pd.DataFrame(KNNImputer(n_neighbors=5).fit_transform(Z), columns=bloc_fin, index=df.index)
df[bloc_fin] = sc_knn.inverse_transform(Z)
print("Manquants restants sur le bloc financier :", df[bloc_fin].isna().sum().sum())

Manquants restants sur le bloc financier : 0


## 7. Ratios financiers et évolution du CA

- `RGI` (chiffre d'affaires) est nul sur 268 lignes donc division impossible ;
- quand `RGI` est proche de 0 sans l'être, le ratio explose : un FDR de 2,3 M rapporté à un CA
  de 155 DH donne **5,4 millions de « jours »**.

Un flag `_NC` par ratio impossible à calculer, et un clip pour les valeurs anormales.

In [ ]:
def sd(n, d):
    return n / d.replace(0, np.nan)

R = {}
R['RA1_FDR']    = df['MC'] - df['CW']
R['RA3_BFDR']   = df['ET'] - df['MP']
R['RA6_TN']     = df['FF'] - df['MT']
R['RA2_FDR_j']  = sd(R['RA1_FDR'],  df['RGI']) * 360
R['RA4_BFDR_j'] = sd(R['RA3_BFDR'], df['RGI']) * 360
R['RA5_Couv']   = sd(R['RA1_FDR'],  R['RA3_BFDR']) * 100
R['RA7_TN_j']   = sd(R['RA6_TN'],   df['RGI']) * 360
R['RA8_CP_CapPerm']  = sd(df['LP'],  df['MC'])  * 100
R['RA9_CP_Bilan']    = sd(df['LP'],  df['FI'])  * 100
R['RA10_DLMT_CAF']   = sd(df['LV'],  df['HA5']) * 100
R['RA11_ChFin_CA']   = sd(df['JCX'], df['RGI']) * 100
R['RA12_Immo_Bilan'] = sd(df['CW'],  df['FI'])  * 100
R['RA13_Amort_Immo'] = sd(df['CV'],  df['CU'])  * 100
R['RA14_BN_CA']      = sd(df['KW'],  df['RGI']) * 100
R['RA15_RN_CP']      = sd(df['KW'],  df['LP'])  * 100
R['RA16_CAF_CA']     = sd(df['HA5'], df['RGI']) * 100
R['RA17_Clients_j']  = sd(df['DU'],  df['RGI']) * 360
R['RA18_Fourn_j']    = sd(df['MD'],  df['HDG']) * 360
R['RA19_Stocks_j']   = sd(df['DO'],  df['RGI']) * 360
R['EVOL_CA_N_N1']    = sd(df['RGI'] - df['GI2'], df['GI2']) * 100
R['EVOL_CA_N1_N2']   = sd(df['GI2'] - df['GI3'], df['GI3']) * 100

ratios = pd.DataFrame(R, index=df.index)
print("Ratios non calculables:")
print(ratios.isna().sum()[ratios.isna().sum() > 0].sort_values(ascending=False).head())

flags = pd.DataFrame({f"FLAG_{c}_NC": ratios[c].isna().astype(int) for c in ratios.columns}, index=df.index)
for c in ratios.columns:                      
    ratios[c] = ratios[c].clip(-1000, 1000) if c.endswith('_j') else ratios[c].clip(-500, 500)

df = pd.concat([df, ratios, flags], axis=1).copy()
print("\n", df.shape)

Ratios non calculables (division par zéro) :
RA2_FDR_j         278
RA4_BFDR_j        278
RA19_Stocks_j     278
RA7_TN_j          278
RA17_Clients_j    278
dtype: int64

 (838, 367)


## 8. Encodage des variables catégorielles

| Variable | Modalités | Méthode |
|---|---|---|
| `REGION_RC`, `REGION_TIERS`, `CENTRE_AFFAIRE` | 13 / 13 / 8 | **one-hot**, `drop_first=True` |
| `VILLE_RC`, `VILLE_TIERS` | 52 / 59 | **frequency encoding** |

In [127]:
low  = [c for c in ['REGION_RC','REGION_TIERS','CENTRE_AFFAIRE'] if c in df.columns]
high = [c for c in ['VILLE_RC','VILLE_TIERS'] if c in df.columns]

df = pd.get_dummies(df, columns=low, drop_first=True)
VILLE = df[high].copy()
df = df.drop(columns=high)

y = df["DEFAUT"].values
X = df.drop(columns=["DEFAUT"])
X[X.select_dtypes(include='bool').columns] = X.select_dtypes(include='bool').astype(int)

print(f"Jeu final : {X.shape[0]} lignes x {X.shape[1]} variables | défaut : {y.mean():.3f}")

Jeu final : 838 lignes x 198 variables | défaut : 0.186


## 9. Protocole d'évaluation

**Validation croisée stratifiée à 5 plis.**

**Aucune fuite.** Toute transformation *estimée* est ajustée sur le train seul : frequency
encoding, winsorisation, imputation, standardisation, et le seuil de décision.

**Déséquilibre.** Pas de SMOTE, pondération de la classe minoritaire, plus stable à cet
effectif, et appliquée aux quatre modèles pour que la comparaison reste équitable.

**Seuil de décision : indice de Youden.**
$$J = \text{sensibilité} + \text{spécificité} - 1 = \text{TPR} - \text{FPR}$$
C'est le point de la courbe ROC le plus éloigné de la diagonale : il équilibre rappel et AUC

> **Le seuil ne peut pas être calibré en échantillon.** Un Random Forest sépare presque
> parfaitement son propre train : le seuil qui y paraît optimal (0,494) est bien trop haut en
> test et le rappel tombe à **0,263**. Calibré sur des prédictions *out-of-fold* d'une CV
> interne à 3 plis, le seuil descend à 0,292 et le rappel atteint **0,667** — 40 points de
> gain. Les modèles à faible variance sont peu affectés, ce qui rend l'erreur facile à manquer.

In [147]:
class DNN(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, 24), nn.BatchNorm1d(24), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(24, 1))
    def forward(self, x): return self.net(x)

def fit_dnn(Xtr, ytr, pw, epochs=250):
    m = DNN(Xtr.shape[1])
    crit = nn.BCEWithLogitsLoss(pos_weight=pw)
    opt  = torch.optim.Adam(m.parameters(), lr=2e-3, weight_decay=1e-3)
    Xt = torch.tensor(Xtr, dtype=torch.float32)
    yt = torch.tensor(ytr, dtype=torch.float32).unsqueeze(1)
    for _ in range(epochs):
        m.train(); opt.zero_grad(); crit(m(Xt), yt).backward(); opt.step()
    return m

def proba_dnn(m, Xte):
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(torch.tensor(Xte, dtype=torch.float32))).numpy().ravel()

def seuil_youden(y_true, proba):
    fpr, tpr, thr = roc_curve(y_true, proba)
    return thr[np.argmax(tpr - fpr)]

def preparer_pli(tr, te):
    # Applique au pli toutes les transformations ajustées sur le train seul
    Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()

    for col in VILLE.columns:                                  # frequency encoding
        freq = VILLE.iloc[tr][col].value_counts(normalize=True)
        Xtr[col+"_FREQ"] = VILLE.iloc[tr][col].map(freq).values
        Xte[col+"_FREQ"] = VILLE.iloc[te][col].map(freq).fillna(0).values

    for c in [c for c in Xtr.columns if Xtr[c].nunique() > 10]:  # winsorisation 1%/99%
        lo, hi = Xtr[c].quantile(0.01), Xtr[c].quantile(0.99)
        Xtr[c] = Xtr[c].clip(lo, hi); Xte[c] = Xte[c].clip(lo, hi)

    imp = SimpleImputer(strategy="median")
    cols = Xtr.columns
    Xtr = pd.DataFrame(imp.fit_transform(Xtr), columns=cols)
    Xte = pd.DataFrame(imp.transform(Xte),     columns=cols)

    sc = StandardScaler()
    return Xtr, Xte, sc.fit_transform(Xtr), sc.transform(Xte)

# Les plis ne dépendent pas des hyperparamètres : on les prépare une fois pour toutes.
SPLITS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(X, y))
PLIS   = [preparer_pli(tr, te) for tr, te in SPLITS]
INNER  = [list(StratifiedKFold(3, shuffle=True, random_state=7).split(p[0], y[tr]))
          for p, (tr, te) in zip(PLIS, SPLITS)]
print(f"{len(PLIS)} plis préparés | {PLIS[0][0].shape[1]} variables après encodage des villes")

5 plis préparés | 200 variables après encodage des villes


## 10. Comparaison des quatre modèles

Pour chaque pli externe : entraînement des 4 modèles, puis calibrage du seuil par CV interne.
Le score composite $\frac{1}{2}(\text{AUC} + \text{rappel})$ arbitre entre pouvoir discriminant
et détection.

In [130]:
def construire(nom, y_fit):
    spw = (y_fit == 0).sum() / (y_fit == 1).sum()
    if nom == "Régression Logistique":
        return LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1, random_state=42)
    if nom == "Random Forest":
        return RandomForestClassifier(n_estimators=500, min_samples_leaf=3, max_features="sqrt",
                                      class_weight="balanced", random_state=42, n_jobs=-1)
    if nom == "XGBoost":
        return XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.05, subsample=0.8,
                             colsample_bytree=0.8, reg_lambda=2.0, scale_pos_weight=spw,
                             eval_metric="logloss", random_state=42, n_jobs=-1)
    raise ValueError(nom)

SK = ["Régression Logistique", "Random Forest", "XGBoost"]
MODELES = SK + ["Réseau de neurones (DNN)"]

res    = {m: [] for m in MODELES}
seuils = {m: [] for m in MODELES}

for (Xtr, Xte, Xtr_s, Xte_s), (tr, te), inner in zip(PLIS, SPLITS, INNER):
    ytr, yte = y[tr], y[te]
    pw = torch.tensor([(ytr == 0).sum() / (ytr == 1).sum()], dtype=torch.float32)

    # --- Seuil : prédictions out-of-fold d'une CV interne à 3 plis ---
    oof = {m: np.zeros(len(ytr)) for m in MODELES}
    for i, j in inner:
        yi = ytr[i]
        for nom in SK:
            Xa, Xb = (Xtr_s[i], Xtr_s[j]) if nom == "Régression Logistique" else (Xtr.iloc[i], Xtr.iloc[j])
            oof[nom][j] = construire(nom, yi).fit(Xa, yi).predict_proba(Xb)[:, 1]
        pw_i = torch.tensor([(yi == 0).sum() / (yi == 1).sum()], dtype=torch.float32)
        oof["Réseau de neurones (DNN)"][j] = proba_dnn(fit_dnn(Xtr_s[i], yi, pw_i), Xtr_s[j])

    # --- Modèles entraînés sur tout le train, évalués sur le test ---
    p_test = {}
    for nom in SK:
        Xa, Xb = (Xtr_s, Xte_s) if nom == "Régression Logistique" else (Xtr, Xte)
        p_test[nom] = construire(nom, ytr).fit(Xa, ytr).predict_proba(Xb)[:, 1]
    p_test["Réseau de neurones (DNN)"] = proba_dnn(fit_dnn(Xtr_s, ytr, pw), Xte_s)

    for nom in MODELES:
        s = seuil_youden(ytr, oof[nom])
        pred = (p_test[nom] >= s).astype(int)
        auc = roc_auc_score(yte, p_test[nom]); rec = recall_score(yte, pred, zero_division=0)
        res[nom].append([auc, average_precision_score(yte, p_test[nom]), accuracy_score(yte, pred),
                         precision_score(yte, pred, zero_division=0), rec,
                         f1_score(yte, pred, zero_division=0), (auc + rec) / 2])
        seuils[nom].append(s)

COLS = ["AUC","PR-AUC","Accuracy","Precision","Recall","F1","Score AUC+Recall"]
tab = pd.DataFrame({n: np.array(res[n]).mean(0) for n in MODELES}, index=COLS).T
tab["Seuil moyen"] = [np.mean(seuils[n]) for n in MODELES]
tab.round(3)

,AUC,PR-AUC,Accuracy,Precision,Recall,F1,Score AUC+Recall,Seuil moyen
Régression Logistique,0.643,0.336,0.654,0.284,0.557,0.375,0.600,0.477
Random Forest,0.684,0.405,0.655,0.286,0.564,0.379,0.624,0.396
XGBoost,0.688,0.404,0.647,0.301,0.621,0.399,0.655,0.222
Réseau de neurones (DNN),0.641,0.370,0.650,0.284,0.508,0.350,0.574,0.242


In [131]:
ecarts = pd.DataFrame({n: np.array(res[n]).std(0) for n in MODELES}, index=COLS).T
print("Écarts-types inter-plis :")
display(ecarts.round(3))

best = tab["Score AUC+Recall"].idxmax()
print(f"\n>> Meilleur compromis AUC/rappel : {best}")
print(f"   AUC {tab.loc[best,'AUC']:.3f} | rappel {tab.loc[best,'Recall']:.3f} "
      f"| F1 {tab.loc[best,'F1']:.3f} | seuil {tab.loc[best,'Seuil moyen']:.3f}")

Écarts-types inter-plis :


,AUC,PR-AUC,Accuracy,Precision,Recall,F1,Score AUC+Recall
Régression Logistique,0.021,0.034,0.031,0.022,0.051,0.022,0.032
Random Forest,0.026,0.051,0.028,0.013,0.046,0.006,0.034
XGBoost,0.013,0.029,0.079,0.039,0.096,0.026,0.046
Réseau de neurones (DNN),0.033,0.047,0.094,0.043,0.137,0.028,0.071



>> Meilleur compromis AUC/rappel : XGBoost
   AUC 0.688 | rappel 0.621 | F1 0.399 | seuil 0.222


**Lecture attendue.** XGBoost obtient la meilleure AUC, mais son rappel est plus instable
(écart-type ≈ 0,105 contre ≈ 0,039 pour le Random Forest). À cet effectif, cette variance
plaide pour le **Random Forest**, meilleur compromis d'ensemble.

Le réseau de neurones reste en retrait : ~670 observations d'entraînement par pli ne suffisent
pas à exploiter sa capacité d'approximation.

## 11. Recherche d'hyperparamètres

Grille évaluée avec le protocole complet (5 plis externes + 3 plis internes pour le seuil).
Cette cellule est la plus longue à exécuter — de l'ordre de quelques minutes.

In [132]:
def scorer(builder):
    A, R = [], []
    for (Xtr, Xte, _, _), (tr, te), inner in zip(PLIS, SPLITS, INNER):
        ytr, yte = y[tr], y[te]
        oof = np.zeros(len(ytr))
        for i, j in inner:
            oof[j] = builder(ytr[i]).fit(Xtr.iloc[i], ytr[i]).predict_proba(Xtr.iloc[j])[:, 1]
        s = seuil_youden(ytr, oof)
        p = builder(ytr).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
        A.append(roc_auc_score(yte, p))
        R.append(recall_score(yte, (p >= s).astype(int), zero_division=0))
    a, r = np.mean(A), np.mean(R)
    return a, r, (a + r) / 2

grille = []
for nt, msl in itertools.product([300, 500], [1, 3, 5]):
    b = lambda yy, nt=nt, msl=msl: RandomForestClassifier(
        n_estimators=nt, min_samples_leaf=msl, max_features="sqrt",
        class_weight="balanced", random_state=42, n_jobs=-1)
    a, r, c = scorer(b)
    grille.append(["Random Forest", f"n_estimators={nt}, min_samples_leaf={msl}", a, r, c])

for md_, rl in itertools.product([2, 3, 4], [1.0, 3.0]):
    b = lambda yy, md_=md_, rl=rl: XGBClassifier(
        n_estimators=400, max_depth=md_, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
        reg_lambda=rl, scale_pos_weight=(yy == 0).sum() / (yy == 1).sum(),
        eval_metric="logloss", random_state=42, n_jobs=-1)
    a, r, c = scorer(b)
    grille.append(["XGBoost", f"max_depth={md_}, reg_lambda={rl}", a, r, c])

grille = pd.DataFrame(grille, columns=["Modèle","Configuration","AUC","Recall","Score"])
grille.sort_values("Score", ascending=False).round(3)

,Modèle,Configuration,AUC,Recall,Score
6,XGBoost,"max_depth=2, reg_lambda=1.0",0.690,0.617,0.654
0,Random Forest,"n_estimators=300, min_samples_leaf=1",0.687,0.616,0.651
7,XGBoost,"max_depth=2, reg_lambda=3.0",0.683,0.616,0.650
10,XGBoost,"max_depth=4, reg_lambda=1.0",0.701,0.596,0.648
3,Random Forest,"n_estimators=500, min_samples_leaf=1",0.684,0.603,0.644
11,XGBoost,"max_depth=4, reg_lambda=3.0",0.698,0.583,0.641
9,XGBoost,"max_depth=3, reg_lambda=3.0",0.692,0.578,0.635
2,Random Forest,"n_estimators=300, min_samples_leaf=5",0.684,0.584,0.634
4,Random Forest,"n_estimators=500, min_samples_leaf=3",0.684,0.564,0.624
1,Random Forest,"n_estimators=300, min_samples_leaf=3",0.685,0.558,0.622


> **Réserve.** Si une configuration affiche un rappel très supérieur à ses voisines alors que
> son AUC bouge à peine, il s'agit probablement de bruit d'échantillonnage : à 838 observations,
> le rappel est bien plus volatil que l'AUC. Confirmer sur plusieurs graines avant de retenir
> une telle configuration.

## 12. Réduction de dimensionnalité

198 variables pour 838 observations, soit environ **4 lignes par variable** : le rapport est
faible et justifie d'examiner si une réduction de dimension améliore la généralisation.

Quatre stratégies sont comparées, de nature très différente :

| Stratégie | Supervisée ? | Principe |
|---|---|---|
| **PLS** | oui | composantes maximisant la covariance avec la cible |
| **PCA** | non | composantes maximisant la variance expliquée |
| **Top-$k$ par importance** | oui | conserve les $k$ variables les mieux notées par un Random Forest |
| **Top-$k$ par corrélation** | oui | conserve les $k$ variables les plus corrélées à la cible, après élimination des redondances |

> **Toute sélection supervisée doit être refaite dans chaque pli.** Choisir les variables sur
> l'ensemble du jeu puis évaluer en validation croisée revient à laisser le test influencer la
> sélection : le score obtenu est optimiste et ne se reproduit pas. Les fonctions ci-dessous
> ajustent donc PLS, PCA, importances et corrélations **sur le train du pli uniquement**.

In [133]:
def _cv_reduction(transformer_fit, make_model, standardise=True):
    # Ossature commune : le reducteur est ajuste sur le train du pli, jamais sur le test.
    A, R = [], []
    for (Xtr, Xte, Xtr_s, Xte_s), (tr, te), inner in zip(PLIS, SPLITS, INNER):
        ytr, yte = y[tr], y[te]
        Atr, Ate = (Xtr_s, Xte_s) if standardise else (Xtr.values, Xte.values)

        oof = np.zeros(len(ytr))                       # seuil : CV interne
        for i, j in inner:
            f = transformer_fit(Atr[i], ytr[i])
            m = make_model(ytr[i]).fit(f(Atr[i]), ytr[i])
            oof[j] = m.predict_proba(f(Atr[j]))[:, 1]
        s = seuil_youden(ytr, oof)

        f = transformer_fit(Atr, ytr)
        m = make_model(ytr).fit(f(Atr), ytr)
        p = m.predict_proba(f(Ate))[:, 1]
        A.append(roc_auc_score(yte, p))
        R.append(recall_score(yte, (p >= s).astype(int), zero_division=0))
    a, r = np.mean(A), np.mean(R)
    return a, r, (a + r) / 2

logit = lambda yy: LogisticRegression(max_iter=3000, class_weight="balanced", C=0.1, random_state=42)

def fit_pls(k):
    def _f(A, yy):
        pls = PLSRegression(n_components=k).fit(A, yy)
        return lambda M: pls.transform(M)
    return _f

def fit_pca(k):
    def _f(A, yy):
        pca = PCA(n_components=k, random_state=42).fit(A)
        return lambda M: pca.transform(M)
    return _f

def fit_identite(A, yy):
    return lambda M: M

### 12.1 Régression PLS puis régression logistique

La PLS projette les variables d'origine sur un petit nombre de composantes **décorrélées entre
elles**, construites pour maximiser leur covariance avec la cible. C'est la parade classique à
la colinéarité des ratios de bilan : les composantes étant orthogonales, la matrice
$X^\top X$ redevient bien conditionnée et les coefficients logistiques se stabilisent.

In [134]:
lignes = []
for k in [2, 5, 10, 20, 40]:
    a, r, c = _cv_reduction(fit_pls(k), logit)
    lignes.append([f"PLS {k} composantes", k, a, r, c])

a, r, c = _cv_reduction(fit_identite, logit)
lignes.append(["Aucune réduction (198 var.)", 198, a, r, c])

pls_res = pd.DataFrame(lignes, columns=["Méthode","Dimension","AUC","Recall","Score"])
pls_res.round(3)

,Méthode,Dimension,AUC,Recall,Score
0,PLS 2 composantes,2,0.617,0.526,0.572
1,PLS 5 composantes,5,0.648,0.634,0.641
2,PLS 10 composantes,10,0.638,0.596,0.617
3,PLS 20 composantes,20,0.622,0.603,0.613
4,PLS 40 composantes,40,0.625,0.642,0.633
5,Aucune réduction (198 var.),198,0.643,0.557,0.600


### 12.2 PCA, sélection par importance, sélection par corrélation

La PCA sert de **témoin non supervisé** : elle ignore la cible et ne retient que les directions
de plus forte variance. Comparer PLS et PCA à nombre de composantes égal isole donc l'apport
de la supervision dans la construction des composantes.

In [135]:
def fit_topk_importance(k):
    def _f(A, yy):
        rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                    random_state=42, n_jobs=-1).fit(A, yy)
        idx = np.argsort(rf.feature_importances_)[::-1][:k]
        return lambda M: M[:, idx]
    return _f

def fit_topk_correlation(k, seuil_redondance=0.7):
    def _f(A, yy):
        D = pd.DataFrame(A)
        c = D.corrwith(pd.Series(yy, index=D.index)).abs().fillna(0).sort_values(ascending=False)
        C = D.corr().abs()
        gardees = []
        for v in c.index:
            if len(gardees) >= k: break
            if all(C.loc[v, g] <= seuil_redondance for g in gardees):
                gardees.append(v)
        idx = np.array(gardees)
        return lambda M: M[:, idx]
    return _f

lignes = []
for k in [10, 40]:
    a, r, c = _cv_reduction(fit_pca(k), logit)
    lignes.append([f"PCA {k} comp. (non supervisée)", k, a, r, c])
for k in [20, 50]:
    a, r, c = _cv_reduction(fit_topk_importance(k), logit)
    lignes.append([f"Top-{k} par importance", k, a, r, c])
for k in [20]:
    a, r, c = _cv_reduction(fit_topk_correlation(k), logit)
    lignes.append([f"Top-{k} par corrélation", k, a, r, c])

autres = pd.DataFrame(lignes, columns=["Méthode","Dimension","AUC","Recall","Score"])
autres.round(3)

,Méthode,Dimension,AUC,Recall,Score
0,PCA 10 comp. (non supervisée),10,0.579,0.391,0.485
1,PCA 40 comp. (non supervisée),40,0.613,0.469,0.541
2,Top-20 par importance,20,0.618,0.546,0.582
3,Top-50 par importance,50,0.661,0.686,0.674
4,Top-20 par corrélation,20,0.626,0.405,0.515


### 12.3 Reduction modèle ensembliste


In [136]:
rf_builder = lambda yy: RandomForestClassifier(n_estimators=500, min_samples_leaf=3,
                                               max_features="sqrt", class_weight="balanced",
                                               random_state=42, n_jobs=-1)
lignes = []
for k in [20, 50]:
    a, r, c = _cv_reduction(fit_topk_importance(k), rf_builder, standardise=False)
    lignes.append([f"Random Forest, top-{k} par importance", k, a, r, c])
a, r, c = _cv_reduction(fit_identite, rf_builder, standardise=False)
lignes.append(["Random Forest, toutes variables", 198, a, r, c])

rf_red = pd.DataFrame(lignes, columns=["Méthode","Dimension","AUC","Recall","Score"])
rf_red.round(3)

,Méthode,Dimension,AUC,Recall,Score
0,"Random Forest, top-20 par importance",20,0.694,0.546,0.620
1,"Random Forest, top-50 par importance",50,0.695,0.520,0.607
2,"Random Forest, toutes variables",198,0.684,0.564,0.624


In [137]:
synthese = pd.concat([
    pls_res.assign(Modèle="Régression logistique"),
    autres.assign(Modèle="Régression logistique"),
    rf_red.assign(Modèle="Random Forest"),
], ignore_index=True)[["Modèle","Méthode","Dimension","AUC","Recall","Score"]]

synthese.sort_values(["Modèle","AUC"], ascending=[True, False]).round(3)

,Modèle,Méthode,Dimension,AUC,Recall,Score
12,Random Forest,"Random Forest, top-50 par importance",50,0.695,0.520,0.607
11,Random Forest,"Random Forest, top-20 par importance",20,0.694,0.546,0.620
13,Random Forest,"Random Forest, toutes variables",198,0.684,0.564,0.624
9,Régression logistique,Top-50 par importance,50,0.661,0.686,0.674
1,Régression logistique,PLS 5 composantes,5,0.648,0.634,0.641
5,Régression logistique,Aucune réduction (198 var.),198,0.643,0.557,0.600
2,Régression logistique,PLS 10 composantes,10,0.638,0.596,0.617
10,Régression logistique,Top-20 par corrélation,20,0.626,0.405,0.515
4,Régression logistique,PLS 40 composantes,40,0.625,0.642,0.633
3,Régression logistique,PLS 20 composantes,20,0.622,0.603,0.613


## 13. Modèle final et importance des variables

In [138]:
imp_folds = []
for (Xtr, _, _, _), (tr, te) in zip(PLIS, SPLITS):
    rf = construire("Random Forest", y[tr]).fit(Xtr, y[tr])
    imp_folds.append(pd.Series(rf.feature_importances_, index=Xtr.columns))

importance = pd.concat(imp_folds, axis=1).mean(axis=1).sort_values(ascending=False)
importance.head(15).round(4).to_frame("Importance moyenne")

,Importance moyenne
AGE_ENTREPRISE_ANS,0.0376
EVOL_CA_N_N1,0.0282
CODE_SECT_ACTIVITE,0.0247
CODE_SSBRANCHE_ACT,0.0222
GI3,0.0211
TAUX_INTERET,0.0209
ID_IFP,0.0201
NBRE_EMPLOI_ACREER,0.0192
RA5_Couv,0.0185
MOIS_ACCORD,0.0178


In [139]:
# Matrice de confusion du modèle retenu, au seuil calibré
Xtr, Xte, _, _ = PLIS[0]
tr, te = SPLITS[0]
ytr, yte = y[tr], y[te]

oof = np.zeros(len(ytr))
for i, j in INNER[0]:
    oof[j] = construire("Random Forest", ytr[i]).fit(Xtr.iloc[i], ytr[i]).predict_proba(Xtr.iloc[j])[:, 1]
s = seuil_youden(ytr, oof)

p = construire("Random Forest", ytr).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
cm = confusion_matrix(yte, (p >= s).astype(int))
print(f"Seuil retenu : {s:.3f}\n")
print(pd.DataFrame(cm, index=["Réel : sain","Réel : défaut"],
                   columns=["Prédit : sain","Prédit : défaut"]))
print(f"\nDéfauts détectés : {cm[1,1]}/{cm[1].sum()} | fausses alertes : {cm[0,1]}")

Seuil retenu : 0.396

               Prédit : sain  Prédit : défaut
Réel : sain               96               41
Réel : défaut             14               17

Défauts détectés : 17/31 | fausses alertes : 41
